# Трек B: проверка воспроизводимости рук в COMBO
Вход: `ts-code`, `ts-models`, данные, `ts-artifacts-dets` (исходный `ft_yolo__v1` для сверки) и `throddy/ts-artifacts-ov-pseudo` (`rowfeats/ov_counts__v1` для конфига COMBO). GPU включён, интернет выключен.

Шаги: заново считаем `ft_yolo__v1` весами из `ts-models` (FP32, как в карточке артефакта), сверяем с исходным файлом, собираем предсказание по `configs/submissions/COMBO.yaml`. Колонка hands результата сравнивается с `submissions/COMBO.csv` локально: ожидается 0 изменённых строк.

In [ ]:
# Подготовка: код из датасета ts-code, данные соревнования, веса и колёса из ts-models. Интернет выключен.
import importlib.util, os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path('/kaggle/input')
code = [p.parent for p in INPUT.rglob('run.py') if (p.parent / 'ts' / '__init__.py').is_file()]
data = [p.parent for p in INPUT.rglob('train_labels.csv')
        if (p.parent / 'tables.csv').is_file() and (p.parent / 'data_card.json').is_file()]
models = [p.parent for p in INPUT.rglob('yolo11x.pt')]
assert len(code) == 1, f'Подключите датасет ts-code (найдено {len(code)})'
assert len(data) == 1, f'Подключите данные соревнования (найдено {len(data)})'
assert len(models) == 1, f'Подключите датасет ts-models (найдено {len(models)})'
REPO, MODELS = Path('/kaggle/working/table-state'), models[0]
shutil.copytree(code[0], REPO, dirs_exist_ok=True)
os.environ['TS_DATA'] = str(data[0])
os.environ['TS_ARTIFACTS'] = '/kaggle/working/artifacts'
os.environ['YOLO_OFFLINE'] = 'true'
os.chdir(REPO)

# Колёса ставим без сети и без зависимостей: torch, torchvision и opencv уже есть в образе.
wheels = sorted((MODELS / 'wheels').glob('*.whl'))
need = [w for w in wheels if not (w.name.startswith('nvidia_ml_py') and importlib.util.find_spec('pynvml'))]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps', *map(str, need)], check=True)
import torch, ultralytics
print('код:', (REPO / 'GIT_SHA').read_text().strip(), '| данные:', data[0], '| веса:', MODELS)
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| GPU:',
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ, включите GPU в настройках')


In [ ]:
import shutil
ART = Path('/kaggle/working/artifacts')
(ART / 'rowfeats').mkdir(parents=True, exist_ok=True)
for f in INPUT.rglob('ov_counts__v1.*'):
    shutil.copy(f, ART / 'rowfeats' / f.name)
ORIGINAL = next(INPUT.rglob('ft_yolo__v1.parquet'))
print(sorted(p.name for p in (ART / 'rowfeats').iterdir()), '| исходный файл:', ORIGINAL)

In [ ]:
!python tracks/det/infer.py --source ft_yolo --weights {MODELS}/ft_yolo_v1.pt

In [ ]:
import pandas as pd, numpy as np
new = pd.read_parquet(ART / 'dets' / 'ft_yolo__v1.parquet')
old = pd.read_parquet(ORIGINAL)
print('детекций: исходный', len(old), '| пересчёт', len(new), '| det_id совпали:', bool(set(old.det_id) == set(new.det_id)))
m = old.merge(new, on='det_id', suffixes=('_old', '_new'))
print('общих det_id:', len(m), '| классы совпали:', bool((m.class_name_old == m.class_name_new).all()))
print('макс. расхождение confidence:', float((m.confidence_old - m.confidence_new).abs().max()))
print('макс. расхождение координат, px:', float(np.abs(m[[c + '_old' for c in ['x1', 'y1', 'x2', 'y2']]].to_numpy() - m[[c + '_new' for c in ['x1', 'y1', 'x2', 'y2']]].to_numpy()).max()))

In [ ]:
!python tracks/core/build_pred.py --name COMBO_repro --config configs/submissions/COMBO.yaml

In [ ]:
!ls -la /kaggle/working/artifacts/dets /kaggle/working/artifacts/preds && rm -rf /kaggle/working/table-state